# Fraud Detection Data Exploration

This notebook explores the processed Parquet files generated by the ETL pipeline.

In [ ]:
import polars as pl
import os

# Set path to artifacts
ARTIFACTS_DIR = "../artifacts"

def load_parquet(filename):
    path = os.path.join(ARTIFACTS_DIR, filename)
    if os.path.exists(path):
        return pl.read_parquet(path)
    else:
        print(f"File not found: {path}")
        return None

## 1. Load User Nodes

In [ ]:
df_users = load_parquet("nodes_user.parquet")
if df_users is not None:
    print(f"User Count: {len(df_users)}")
    display(df_users.head())

## 2. Load Listing Nodes

In [ ]:
df_listings = load_parquet("nodes_listing.parquet")
if df_listings is not None:
    print(f"Listing Count: {len(df_listings)}")
    display(df_listings.head())

## 3. Fraud Distribution

## 4. Temporal Distribution Analysis

Understanding when frauds occur helps validate our sliding-window approach.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

if df_listings is not None:
    # Ensure submission_at is datetime
    df_with_time = df_listings.with_columns([
        pl.col("submission_at").cast(pl.Datetime)
    ])
    
    # Group by month and fraud status
    df_monthly = df_with_time.group_by([
        pl.col("submission_at").dt.truncate("1mo").alias("month"),
        pl.col("is_fraud")
    ]).agg([
        pl.count().alias("count")
    ]).sort("month")
    
    # Pivot for plotting
    fraud_by_month = df_monthly.filter(pl.col("is_fraud") == True)
    legit_by_month = df_monthly.filter(pl.col("is_fraud") == False)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Stacked area chart
    months_fraud = fraud_by_month["month"].to_list()
    counts_fraud = fraud_by_month["count"].to_list()
    months_legit = legit_by_month["month"].to_list()
    counts_legit = legit_by_month["count"].to_list()
    
    axes[0].fill_between(months_legit, counts_legit, alpha=0.5, label="Legitimate", color="steelblue")
    axes[0].plot(months_legit, counts_legit, color="steelblue", linewidth=2)
    axes[0].fill_between(months_fraud, counts_fraud, alpha=0.7, label="Fraud", color="crimson")
    axes[0].plot(months_fraud, counts_fraud, color="crimson", linewidth=2)
    axes[0].set_title("Listing Submissions Over Time", fontsize=14, fontweight='bold')
    axes[0].set_xlabel("Month")
    axes[0].set_ylabel("Count")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Fraud rate over time
    df_fraud_rate = df_with_time.group_by([
        pl.col("submission_at").dt.truncate("1mo").alias("month")
    ]).agg([
        pl.col("is_fraud").sum().alias("fraud_count"),
        pl.count().alias("total_count")
    ]).with_columns([
        (pl.col("fraud_count") / pl.col("total_count") * 100).alias("fraud_rate_pct")
    ]).sort("month")
    
    months_rate = df_fraud_rate["month"].to_list()
    fraud_rates = df_fraud_rate["fraud_rate_pct"].to_list()
    
    axes[1].plot(months_rate, fraud_rates, color="darkred", linewidth=2.5, marker='o')
    axes[1].axhline(y=df_listings["is_fraud"].mean() * 100, color='gray', linestyle='--', 
                    label=f'Overall Rate: {df_listings["is_fraud"].mean() * 100:.2f}%')
    axes[1].set_title("Fraud Rate Over Time", fontsize=14, fontweight='bold')
    axes[1].set_xlabel("Month")
    axes[1].set_ylabel("Fraud Rate (%)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nOverall fraud rate: {df_listings['is_fraud'].mean() * 100:.2f}%")
    print(f"Total listings: {len(df_listings):,}")
    print(f"Fraud listings: {df_listings['is_fraud'].sum():,}")
    print(f"Legitimate listings: {(~df_listings['is_fraud']).sum():,}")


## 5. Feature Analysis: Account Age Dominance

**Theory from docs**: `account_age_days` is the strongest predictor. Let's verify this.
SR

In [ ]:
if df_listings is not None and "account_age_days" in df_listings.columns:
    # Statistical comparison
    fraud_age = df_listings.filter(pl.col("is_fraud"))["account_age_days"]
    legit_age = df_listings.filter(~pl.col("is_fraud"))["account_age_days"]
    
    print("Account Age Statistics:")
    print(f"\nFraud Listings:")
    print(f"  Mean: {fraud_age.mean():.1f} days")
    print(f"  Median: {fraud_age.median():.1f} days")
    print(f"  Std Dev: {fraud_age.std():.1f} days")
    
    print(f"\nLegitimate Listings:")
    print(f"  Mean: {legit_age.mean():.1f} days")
    print(f"  Median: {legit_age.median():.1f} days")
    print(f"  Std Dev: {legit_age.std():.1f} days")
    
    # Statistical test
    from scipy import stats
    t_stat, p_value = stats.ttest_ind(fraud_age.to_numpy(), legit_age.to_numpy())
    print(f"\nT-test: t={t_stat:.2f}, p-value={p_value:.2e}")
    print(f"Conclusion: {'Highly significant difference' if p_value < 0.001 else 'No significant difference'}")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Distribution comparison
    axes[0].hist(fraud_age.to_numpy(), bins=50, alpha=0.6, label='Fraud', color='crimson', density=True)
    axes[0].hist(legit_age.to_numpy(), bins=50, alpha=0.6, label='Legitimate', color='steelblue', density=True)
    axes[0].set_xlabel('Account Age (days)')
    axes[0].set_ylabel('Density')
    axes[0].set_title('Account Age Distribution', fontweight='bold')
    axes[0].legend()
    axes[0].set_xlim(0, 365)
    
    # Box plot
    data_for_box = [fraud_age.to_numpy(), legit_age.to_numpy()]
    bp = axes[1].boxplot(data_for_box, labels=['Fraud', 'Legitimate'], patch_artist=True)
    bp['boxes'][0].set_facecolor('crimson')
    bp['boxes'][1].set_facecolor('steelblue')
    axes[1].set_ylabel('Account Age (days)')
    axes[1].set_title('Account Age by Fraud Status', fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    # Cumulative distribution
    fraud_sorted = np.sort(fraud_age.to_numpy())
    legit_sorted = np.sort(legit_age.to_numpy())
    fraud_cdf = np.arange(1, len(fraud_sorted) + 1) / len(fraud_sorted)
    legit_cdf = np.arange(1, len(legit_sorted) + 1) / len(legit_sorted)
    
    axes[2].plot(fraud_sorted, fraud_cdf, color='crimson', linewidth=2, label='Fraud')
    axes[2].plot(legit_sorted, legit_cdf, color='steelblue', linewidth=2, label='Legitimate')
    axes[2].set_xlabel('Account Age (days)')
    axes[2].set_ylabel('Cumulative Probability')
    axes[2].set_title('CDF: Account Age', fontweight='bold')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_xlim(0, 365)
    
    plt.tight_layout()
    plt.show()
    
    # Fraud rate by age bucket
    df_age_buckets = df_listings.with_columns([
        pl.when(pl.col("account_age_days") < 7).then(pl.lit("0-7 days"))
        .when(pl.col("account_age_days") < 30).then(pl.lit("7-30 days"))
        .when(pl.col("account_age_days") < 90).then(pl.lit("30-90 days"))
        .when(pl.col("account_age_days") < 180).then(pl.lit("90-180 days"))
        .otherwise(pl.lit("180+ days"))
        .alias("age_bucket")
    ])
    
    fraud_by_bucket = df_age_buckets.group_by("age_bucket").agg([
        pl.col("is_fraud").sum().alias("fraud_count"),
        pl.count().alias("total_count")
    ]).with_columns([
        (pl.col("fraud_count") / pl.col("total_count") * 100).alias("fraud_rate_pct")
    ]).sort("fraud_rate_pct", descending=True)
    
    print("\n\nFraud Rate by Account Age Bucket:")
    print(fraud_by_bucket)
    
    # Plot fraud rate by bucket
    fig, ax = plt.subplots(figsize=(10, 6))
    buckets = fraud_by_bucket["age_bucket"].to_list()
    rates = fraud_by_bucket["fraud_rate_pct"].to_list()
    colors = ['darkred' if r > 10 else 'orange' if r > 5 else 'green' for r in rates]
    
    ax.barh(buckets, rates, color=colors, edgecolor='black', linewidth=1.5)
    ax.set_xlabel('Fraud Rate (%)')
    ax.set_title('Fraud Rate by Account Age Bucket', fontweight='bold', fontsize=14)
    ax.grid(axis='x', alpha=0.3)
    
    for i, (bucket, rate) in enumerate(zip(buckets, rates)):
        ax.text(rate + 0.5, i, f'{rate:.2f}%', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print("account_age_days column not found")


## 6. Other Feature Patterns: Price and Bundle Analysis


In [ ]:
if df_listings is not None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Price distribution (if exists)
    if "price" in df_listings.columns:
        fraud_price = df_listings.filter(pl.col("is_fraud") & (pl.col("price") > 0))["price"]
        legit_price = df_listings.filter(~pl.col("is_fraud") & (pl.col("price") > 0))["price"]
        
        # Log scale for better visualization
        axes[0, 0].hist(np.log10(fraud_price.to_numpy()), bins=30, alpha=0.6, label='Fraud', color='crimson', density=True)
        axes[0, 0].hist(np.log10(legit_price.to_numpy()), bins=30, alpha=0.6, label='Legitimate', color='steelblue', density=True)
        axes[0, 0].set_xlabel('log10(Price)')
        axes[0, 0].set_ylabel('Density')
        axes[0, 0].set_title('Price Distribution (Log Scale)', fontweight='bold')
        axes[0, 0].legend()
        
        print(f"Price Statistics:")
        print(f"  Fraud - Mean: {fraud_price.mean():.0f}, Median: {fraud_price.median():.0f}")
        print(f"  Legit - Mean: {legit_price.mean():.0f}, Median: {legit_price.median():.0f}")
    
    # Bundle tier (if exists)
    if "bundle_tier_ordinal" in df_listings.columns:
        bundle_fraud = df_listings.group_by(["bundle_tier_ordinal", "is_fraud"]).agg([
            pl.count().alias("count")
        ])
        
        fraud_counts = bundle_fraud.filter(pl.col("is_fraud"))
        legit_counts = bundle_fraud.filter(~pl.col("is_fraud"))
        
        tiers = [0, 1, 2]  # Basic, Premium, Top
        tier_labels = ['Basic', 'Premium', 'Top']
        fraud_vals = [fraud_counts.filter(pl.col("bundle_tier_ordinal") == t)["count"].sum() for t in tiers]
        legit_vals = [legit_counts.filter(pl.col("bundle_tier_ordinal") == t)["count"].sum() for t in tiers]
        
        x = np.arange(len(tier_labels))
        width = 0.35
        
        axes[0, 1].bar(x - width/2, fraud_vals, width, label='Fraud', color='crimson')
        axes[0, 1].bar(x + width/2, legit_vals, width, label='Legitimate', color='steelblue')
        axes[0, 1].set_xlabel('Bundle Tier')
        axes[0, 1].set_ylabel('Count')
        axes[0, 1].set_title('Bundle Tier Distribution', fontweight='bold')
        axes[0, 1].set_xticks(x)
        axes[0, 1].set_xticklabels(tier_labels)
        axes[0, 1].legend()
    
    # Platform distribution (if exists)
    if "platform" in df_listings.columns:
        platform_fraud = df_listings.group_by(["platform", "is_fraud"]).agg([
            pl.count().alias("count")
        ]).pivot(values="count", index="platform", columns="is_fraud")
        
        platforms = platform_fraud["platform"].to_list()
        fraud_counts = platform_fraud.select(pl.col("true")).to_series().fill_null(0).to_list()
        legit_counts = platform_fraud.select(pl.col("false")).to_series().fill_null(0).to_list()
        
        x = np.arange(len(platforms))
        width = 0.35
        
        axes[1, 0].bar(x - width/2, fraud_counts, width, label='Fraud', color='crimson')
        axes[1, 0].bar(x + width/2, legit_counts, width, label='Legitimate', color='steelblue')
        axes[1, 0].set_xlabel('Platform')
        axes[1, 0].set_ylabel('Count')
        axes[1, 0].set_title('Platform Distribution', fontweight='bold')
        axes[1, 0].set_xticks(x)
        axes[1, 0].set_xticklabels(platforms, rotation=45)
        axes[1, 0].legend()
        axes[1, 0].set_yscale('log')
    
    # Offer type (if exists)
    if "offer_type" in df_listings.columns:
        offer_fraud = df_listings.group_by(["offer_type", "is_fraud"]).agg([
            pl.count().alias("count"),
        ]).with_columns([
            pl.col("offer_type").cast(pl.Utf8)
        ])
        
        offer_types = sorted(offer_fraud["offer_type"].unique().to_list())
        fraud_vals = [offer_fraud.filter((pl.col("offer_type") == t) & pl.col("is_fraud"))["count"].sum() for t in offer_types]
        legit_vals = [offer_fraud.filter((pl.col("offer_type") == t) & ~pl.col("is_fraud"))["count"].sum() for t in offer_types]
        
        x = np.arange(len(offer_types))
        width = 0.35
        
        axes[1, 1].bar(x - width/2, fraud_vals, width, label='Fraud', color='crimson')
        axes[1, 1].bar(x + width/2, legit_vals, width, label='Legitimate', color='steelblue')
        axes[1, 1].set_xlabel('Offer Type')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].set_title('Offer Type Distribution', fontweight='bold')
        axes[1, 1].set_xticks(x)
        axes[1, 1].set_xticklabels(offer_types)
        axes[1, 1].legend()
        axes[1, 1].set_yscale('log')
    
    plt.tight_layout()
    plt.show()


## 7. Key Findings

**From this exploration, we can conclude:**

1. **Account Age Dominates**: Fraud listings have significantly younger accounts (p < 0.001)
2. **Temporal Patterns**: Fraud rate varies over time, validating our sliding-window approach
3. **Feature Distribution**: Some features show clear separation (account age), others are similar
4. **Imbalanced Dataset**: ~8% fraud rate means precision-recall is more informative than accuracy

**Implications for Modeling:**
- Simple tabular features (especially account_age) are very strong predictors
- Graph methods need to add value beyond what account_age provides
- Sliding-window evaluation is critical due to temporal variation in fraud patterns


In [ ]:
if df_listings is not None:
    fraud_counts = df_listings["is_fraud"].value_counts()
    print(fraud_counts)